Cell 1 – Imports and paths

In [2]:
# RS_GROUP_SERVER_02_PATH = "/home/arsalan/wsu_grid_rs_grp_mljet/"
RS_GROUP_SERVER_03_PATH = "/mnt/Research/arsalan/wsu_grid_rs_grp_mljet/ml-jet-ext/"
RS_GROUP_SERVER= RS_GROUP_SERVER_03_PATH or RS_GROUP_SERVER_02_PATH

print(RS_GROUP_SERVER)

/mnt/Research/arsalan/wsu_grid_rs_grp_mljet/ml-jet-ext/


In [3]:
from pathlib import Path
import pandas as pd

# Point this to your metadata directory
# If your notebook is already in the metadata dir, you can use Path(".")
METADATA_DIR = Path(RS_GROUP_SERVER + "pTHat50-250_eCM5020_pTHat50-250_eCM5020_pT90-110_R0p4/metadata")  # <-- CHANGE THIS
METADATA_DIR


PosixPath('/mnt/Research/arsalan/wsu_grid_rs_grp_mljet/ml-jet-ext/pTHat50-250_eCM5020_pTHat50-250_eCM5020_pT90-110_R0p4/metadata')

Cell 2 – Collect the per-config summary rows (empty seed)

In [4]:
seed_stat_files = sorted(METADATA_DIR.glob("*_seed_stats.csv"))
seed_stat_files


[PosixPath('/mnt/Research/arsalan/wsu_grid_rs_grp_mljet/ml-jet-ext/pTHat50-250_eCM5020_pTHat50-250_eCM5020_pT90-110_R0p4/metadata/c0005_alpha0p20_q_1p0_unfolded_seed_stats.csv'),
 PosixPath('/mnt/Research/arsalan/wsu_grid_rs_grp_mljet/ml-jet-ext/pTHat50-250_eCM5020_pTHat50-250_eCM5020_pT90-110_R0p4/metadata/c0005_alpha0p20_q_1p5_unfolded_seed_stats.csv'),
 PosixPath('/mnt/Research/arsalan/wsu_grid_rs_grp_mljet/ml-jet-ext/pTHat50-250_eCM5020_pTHat50-250_eCM5020_pT90-110_R0p4/metadata/c0005_alpha0p20_q_2p0_unfolded_seed_stats.csv'),
 PosixPath('/mnt/Research/arsalan/wsu_grid_rs_grp_mljet/ml-jet-ext/pTHat50-250_eCM5020_pTHat50-250_eCM5020_pT90-110_R0p4/metadata/c0005_alpha0p20_q_2p5_unfolded_seed_stats.csv'),
 PosixPath('/mnt/Research/arsalan/wsu_grid_rs_grp_mljet/ml-jet-ext/pTHat50-250_eCM5020_pTHat50-250_eCM5020_pT90-110_R0p4/metadata/c0005_alpha0p20_q_3p0_unfolded_seed_stats.csv'),
 PosixPath('/mnt/Research/arsalan/wsu_grid_rs_grp_mljet/ml-jet-ext/pTHat50-250_eCM5020_pTHat50-250_eCM502

In [6]:
summary_rows = []

for path in seed_stat_files:
    df = pd.read_csv(path)
    
    # summary row: the one with empty/NaN 'seed'
    if "seed" not in df.columns:
        print(f"[WARN] 'seed' column missing in {path.name}, skipping.")
        continue
    
    mask = df["seed"].isna() | (df["seed"].astype(str).str.strip() == "")
    summary = df[mask]
    
    if summary.empty:
        print(f"[WARN] No summary row (empty seed) found in {path.name}, skipping.")
        continue
    
    # take the first such row and remember which config file it came from
    # summary_rows.append(
    #     summary.iloc[0].assign(config_file=path.name)
    # )
    row = summary.iloc[0].copy()        # Series
    row["config_file"] = path.name      # add new field
    summary_rows.append(row)

len(summary_rows)


35

Cell 3 – Build a DataFrame of per-config summaries

In [7]:
if not summary_rows:
    raise RuntimeError("No summary rows collected. Check your METADATA_DIR or file format.")


summary_df = pd.DataFrame(summary_rows)
summary_df.head()


,seed_dir,seed,alpha,q0,n_events,phi_min,phi_max,eta_min,eta_max,pT_min,pT_max,config_file
200,c0005_alpha0p20_q_1p0_unfolded,NaN,0.2,1.0,221933,-3.031136,3.08711,-4.24388,4.83643,0.000284,109.297997,c0005_alpha0p20_q_1p0_unfolded_seed_stats.csv
200,c0005_alpha0p20_q_1p5_unfolded,NaN,0.2,1.5,219116,-3.102975,2.34495,-3.88349,5.66622,0.000237,109.457001,c0005_alpha0p20_q_1p5_unfolded_seed_stats.csv
200,c0005_alpha0p20_q_2p0_unfolded,NaN,0.2,2.0,216107,-3.088435,3.12184,-5.01880,3.95769,0.000261,109.297997,c0005_alpha0p20_q_2p0_unfolded_seed_stats.csv
200,c0005_alpha0p20_q_2p5_unfolded,NaN,0.2,2.5,214388,-3.115265,2.82460,-4.55330,4.49066,0.000206,109.806999,c0005_alpha0p20_q_2p5_unfolded_seed_stats.csv
200,c0005_alpha0p20_q_3p0_unfolded,NaN,0.2,3.0,213220,-2.482206,3.13966,-4.80446,4.54445,0.000517,109.297997,c0005_alpha0p20_q_3p0_unfolded_seed_stats.csv


Cell 4 – Compute global min/max and total events

In [8]:
n_configs = len(summary_df)
total_events = summary_df["n_events"].astype(int).sum()

phi_min_global = summary_df["phi_min"].min()
phi_max_global = summary_df["phi_max"].max()
eta_min_global = summary_df["eta_min"].min()
eta_max_global = summary_df["eta_max"].max()
pT_min_global  = summary_df["pT_min"].min()
pT_max_global  = summary_df["pT_max"].max()

print(f"Configs:        {n_configs}")
print(f"Total events:   {total_events}")
print(f"phi range:      [{phi_min_global:.6g}, {phi_max_global:.6g}]")
print(f"eta range:      [{eta_min_global:.6g}, {eta_max_global:.6g}]")
print(f"pT  range:      [{pT_min_global:.6g}, {pT_max_global:.6g}]")


Configs:        35
Total events:   6953981
phi range:      [-3.14153, 3.14041]
eta range:      [-7.59804, 7.31479]
pT  range:      [1.46383e-05, 109.935]


In [9]:
global_stats = pd.DataFrame(
    [
        {
            "n_configs": n_configs,
            "n_events_total": total_events,
            "phi_min": phi_min_global,
            "phi_max": phi_max_global,
            "eta_min": eta_min_global,
            "eta_max": eta_max_global,
            "pT_min": pT_min_global,
            "pT_max": pT_max_global,
        }
    ]
)

out_path = METADATA_DIR / "dataset_global_stats.csv"
global_stats.to_csv(out_path, index=False)

out_path, global_stats


(PosixPath('/mnt/Research/arsalan/wsu_grid_rs_grp_mljet/ml-jet-ext/pTHat50-250_eCM5020_pTHat50-250_eCM5020_pT90-110_R0p4/metadata/dataset_global_stats.csv'),
    n_configs  n_events_total   phi_min  phi_max  eta_min  eta_max    pT_min  \
 0         35         6953981 -3.141526  3.14041 -7.59804  7.31479  0.000015   
 
        pT_max  
 0  109.934998  )